In [ ]:
import numpy as np
import pandas as pd

from functools import partial
import importlib

import src.data
import src.features
import src.kernel_normalisation
import src.kernels
import src.krr
import src.metrics
import src.multikernel

from src.data import encode_labels, load_data, train_val_split
from src.features import HOG, UniformLBPHistogram
from src.kernel_normalisation import unit_diag
from src.kernels import *
from src.krr import KernelRidgeRegression
from src.metrics import accuracy
from src.multikernel import *

In [2]:
DATA_DIR = "data/"

X_tr, X_te, y_tr_raw = load_data(DATA_DIR)

y_tr, inv_map = encode_labels(y_tr_raw)

n_classes = len(np.unique(y_tr))

### LBP features

In [24]:
lbp = UniformLBPHistogram(cell_size=8, pool="concat", per_channel=True).fit(X_tr)

X_tr_lbp = lbp.transform(X_tr)
X_te_lbp = lbp.transform(X_te)

X_train_lbp, X_val_lbp, y_train_lbp, y_val_lbp = train_val_split(
    X_tr_lbp, y_tr, test_size=0.3, seed=20
)

In [25]:
X_train_lbp.shape

(3500, 2832)

Grid search over kernel weights

In [26]:
gamma = estimate_gamma(X_train_lbp)

lams = [1e-2, 1e-3, 1e-4, 1e-5]

best = {"acc": -1.0, "lam": None, "gamma": None}

for lam in lams:
    kernel_fn = partial(
        gaussian_kernel_matrix,
        gamma=gamma,
    )
    model = KernelRidgeRegression(
        n_classes=n_classes,
        kernel_fn=kernel_fn,
        lam=lam,
    ).fit(X_train_lbp, y_train_lbp, X_val=X_val_lbp, y_val=y_val_lbp)

    pred_val, _ = model.predict(X_val_lbp)
    acc = accuracy(y_val_lbp, pred_val)
    print(f"lam={lam:.1e}, val_acc={acc:.4f}")

    if acc > best["acc"]:
        best = {"acc": acc, "lam": lam, "gamma": gamma}

print("best:", best)

lam=1.0e-02, val_acc=0.3720
lam=1.0e-03, val_acc=0.4527
lam=1.0e-04, val_acc=0.4933
lam=1.0e-05, val_acc=0.4587
best: {'acc': 0.49333333333333335, 'lam': 0.0001, 'gamma': 0.3043090638929237}


### HOG features

In [28]:
hog = HOG(
    image_shape=(3, 32, 32),
    colour_mode="rgb",
    cell_size=8,
    block_size=2,
    block_stride=1,
    num_bins=6,
    signed=False,
).fit(X_tr)

X_tr_hog = hog.transform(X_tr)
X_te_hog = hog.transform(X_te)

X_train_hog, X_val_hog, y_train_hog, y_val_hog = train_val_split(
    X_tr_hog, y_tr, test_size=0.3, seed=20
)

In [29]:
gamma_lbp = estimate_gamma(X_train_lbp)
lbp_gaussian = partial(gaussian_kernel_matrix, gamma=gamma_lbp)

gammas_hog = estimate_chi2_gammas_channel(X_train_hog)
hog_chi2 = partial(chi2_rbf_kernel_matrix_channel, gammas=gammas_hog)

# Two-kernel setup:
specs: list[KernelSpec] = [
    {
        "name": "hog_chi2",
        "Z": X_train_hog,
        "kernel_fn": hog_chi2,
        "diag_fn": unit_diag,
        "normalise": True,
    },
    {
        "name": "lbp_gaussian",
        "Z": X_train_lbp,
        "kernel_fn": lbp_gaussian,
        "diag_fn": unit_diag,
        "normalise": True,
    },
]

out = beta_grid_search_cv_two_kernels(
    specs,
    y_train_hog,
    n_classes=n_classes,
    model="krr",
    betas=np.linspace(0, 1, 21),
    k=5,
    seed=20,
    lam=1e-4,
    verbose=1,
)

print("best_beta:", out["best_beta"], "best_mean_acc:", out["best_mean_acc"])

beta=0.000 mean_acc=0.4617
beta=0.050 mean_acc=0.5517
beta=0.100 mean_acc=0.5683
beta=0.150 mean_acc=0.5769
beta=0.200 mean_acc=0.5800
beta=0.250 mean_acc=0.5820
beta=0.300 mean_acc=0.5869
beta=0.350 mean_acc=0.5883
beta=0.400 mean_acc=0.5900
beta=0.450 mean_acc=0.5886
beta=0.500 mean_acc=0.5903
beta=0.550 mean_acc=0.5897
beta=0.600 mean_acc=0.5871
beta=0.650 mean_acc=0.5851
beta=0.700 mean_acc=0.5849
beta=0.750 mean_acc=0.5846
beta=0.800 mean_acc=0.5791
beta=0.850 mean_acc=0.5769
beta=0.900 mean_acc=0.5740
beta=0.950 mean_acc=0.5683
beta=1.000 mean_acc=0.5489
best_beta: 0.5 best_mean_acc: 0.5902857142857144


## After tuning parameters, train the best model.

### Check the best model

In [30]:
beta = float(np.asarray(out["best_beta"]).item())

# IMPORTANT: use the partial kernels you defined (with gammas)
Kl_tr = lbp_gaussian(X_train_lbp, X_train_lbp)
Kh_tr = hog_chi2(X_train_hog, X_train_hog)

# Match CV: unit-diagonal normalisation per kernel
Kl_tr = normalise_train_gram(Kl_tr, unit_diag(X_train_lbp))
Kh_tr = normalise_train_gram(Kh_tr, unit_diag(X_train_hog))

Ktr = beta * Kl_tr + (1.0 - beta) * Kh_tr

Ks_val = lbp_gaussian(X_val_lbp, X_train_lbp)
Kh_val = hog_chi2(X_val_hog, X_train_hog)

Ks_val = normalise_cross_gram(Ks_val, unit_diag(X_val_lbp), unit_diag(X_train_lbp))
Kh_val = normalise_cross_gram(Kh_val, unit_diag(X_val_hog), unit_diag(X_train_hog))

K_val = beta * Ks_val + (1.0 - beta) * Kh_val

best_model = KernelRidgeRegression(n_classes=n_classes, kernel_fn=None, lam=1e-4).fit(
    None, y_train_hog, K=Ktr
)
yhat_val, _ = best_model.predict(K_star=K_val)
print("val acc:", accuracy(y_val_hog, yhat_val))

val acc: 0.6113333333333333


### Train on full dataset for submission

In [31]:
beta = float(np.asarray(out["best_beta"]).item())

# Re-estimate gammas on FULL training set (not the CV subset)
gamma_lbp = estimate_gamma(X_tr_lbp)
lbp_gaussian = partial(gaussian_kernel_matrix, gamma=gamma_lbp)

gammas_hog = estimate_chi2_gammas_channel(X_tr_hog)
hog_chi2 = partial(chi2_rbf_kernel_matrix_channel, gammas=gammas_hog)

# FULL train Gram
Kl_tr = lbp_gaussian(X_tr_lbp, X_tr_lbp)
Kh_tr = hog_chi2(X_tr_hog, X_tr_hog)

Kl_tr = normalise_train_gram(Kl_tr, unit_diag(X_tr_lbp))
Kh_tr = normalise_train_gram(Kh_tr, unit_diag(X_tr_hog))

Ktr = beta * Kl_tr + (1.0 - beta) * Kh_tr

best_model = KernelRidgeRegression(n_classes=n_classes, kernel_fn=None, lam=1e-4).fit(
    None, y_tr, K=Ktr
)

# TEST x TRAIN cross Gram
Ks_te_tr = lbp_gaussian(X_te_lbp, X_tr_lbp)
Kh_te_tr = hog_chi2(X_te_hog, X_tr_hog)

Ks_te_tr = normalise_cross_gram(Ks_te_tr, unit_diag(X_te_lbp), unit_diag(X_tr_lbp))
Kh_te_tr = normalise_cross_gram(Kh_te_tr, unit_diag(X_te_hog), unit_diag(X_tr_hog))

K_star = beta * Ks_te_tr + (1.0 - beta) * Kh_te_tr

yte_int, _ = best_model.predict(K_star=K_star)
yte = np.array([inv_map[i] for i in yte_int])

sub = pd.DataFrame({"Prediction": yte})
sub.index += 1
sub.to_csv("results/submission.csv", index_label="Id")